# 01 — Data Ingestion

**Goal:** Pull all three raw data sources into Google Drive, filtered to NPM only.

## What this notebook produces
| Output file (in `data/`) | Contents |
|---|---|
| `processed/npm_packages.parquet` | NPM package metadata from Libraries.io |
| `processed/npm_dependencies.parquet` | NPM dependency edges (package → package) |
| `processed/gh_commits.parquet` | GitHub Archive commit events for NPM repos |
| `raw/osv_npm/` | OSV vulnerability JSON files (one per CVE) |
| `processed/npm_maintainers.parquet` | npm registry maintainer + publish-date data |
| `sample/npm_packages_10k.parquet` | Top 10k packages by dependent count (for fast dev) |

## Learning note
Each section has a **Why** block explaining the design decision. Read those before running the cells — they are the most important part of this notebook as a learning exercise.

In [1]:
# ===============================================
# BASIC SETUP: GIT Auth and mounting google drive
# ================================================

from google.colab import drive, userdata
import sys
import os

# Standard mount to access the config file initially
drive.mount('/content/drive')
PROJECT_ROOT = userdata.get('BLAST_RADIUS_PATH')

# Add PROJECT_ROOT to sys.path if it's not already there
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Ensure the utils directory is discoverable if it's directly under PROJECT_ROOT
utils_path = os.path.join(PROJECT_ROOT, 'utils')
if os.path.exists(utils_path) and utils_path not in sys.path:
    sys.path.append(utils_path)

from utils.config import initialize_project
initialize_project()

Mounted at /content/drive
Authenticated as: Ande404
Working Directory: /content/drive/MyDrive/projects/BlastRadius


## 0 — Install dependencies & mount Drive

**Why:** We install at notebook start so the environment is self-contained — anyone with Colab Pro can open this and run it without any prior setup. `gdown` lets us download files from Google Drive by share link, which is useful for the Libraries.io dataset after you've uploaded it.

In [ ]:
!pip install -q pandas pyarrow requests aiohttp tqdm google-cloud-bigquery google-cloud-storage db-dtypes

In [2]:


# Create all subdirectories if they don't exist yet
for subdir in ['data/raw/osv_npm', 'data/processed', 'data/sample']:
    os.makedirs(f'{PROJECT_ROOT}/{subdir}', exist_ok=True)

print(f'Working directory: {PROJECT_ROOT}')
print('Subdirectories created:')
for root, dirs, files in os.walk(PROJECT_ROOT):
    level = root.replace(PROJECT_ROOT, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')

Working directory: /content/drive/MyDrive/projects/BlastRadius
Subdirectories created:
BlastRadius/
  .git/
    info/
    branches/
    hooks/
    refs/
      heads/
      tags/
      remotes/
        origin/
    objects/
      9b/
      09/
      b3/
      7a/
      2d/
      pack/
      info/
      55/
      14/
      16/
    logs/
      refs/
        heads/
        remotes/
          origin/
  notebooks/
    .ipynb_checkpoints/
  utils/
    __pycache__/
  data/
    raw/
      osv_npm/
    processed/
    sample/


## 1 — Authenticate with Google Cloud

**Why:** BigQuery is how we get contributor commit data from the GitHub Archive — it's a public dataset that Google hosts for free (up to 1TB/month of queries). Colab Pro has GCP integration built in, so `authenticate_user()` opens a browser OAuth flow and stores credentials for the session. You won't need a service account key file.

In [5]:
from google.colab import auth
auth.authenticate_user()
print('GCP authentication complete.')

GCP authentication complete.


In [6]:
# --- CONFIGURE THIS ---
# Replace with your Google Cloud project ID (create one free at console.cloud.google.com)
GCP_PROJECT_ID = userdata.get('GCP_BLAST_RADIUS-ID')
# ----------------------

from google.cloud import bigquery
bq = bigquery.Client(project=GCP_PROJECT_ID)
print(f'BigQuery client ready. Project: {GCP_PROJECT_ID}')

BigQuery client ready. Project: blast-radius-494118


## 2 — Libraries.io: NPM packages and dependencies

**Why:** Libraries.io tracks 2.5M+ open-source components across 35 package managers. We use it as our primary source for:
- Package metadata (name, homepage, stars, forks, dependent count)
- Dependency edges (which package depends on which)
- Repository URLs (needed to join with GitHub Archive data)

**How to get the data:**
1. Go to https://kaggle.com/datasets/librariesdotio/libraries-io
2. Download the dataset (requires a free Kaggle account)
3. Upload `projects-1.6.0-2020-01-12.csv` and `dependencies-1.6.0-2020-01-12.csv` to your Google Drive at `BlastRadius/data/raw/`

The cells below read those files and filter them down to NPM only.

In [ ]:
import pandas as pd

# --- Update these filenames to match what you downloaded ---
PROJECTS_CSV  = f'{PROJECT_ROOT}/data/raw/projects-1.6.0-2020-01-12.csv'
DEPS_CSV      = f'{PROJECT_ROOT}/data/raw/dependencies-1.6.0-2020-01-12.csv'
# -----------------------------------------------------------

print('Loading projects CSV (this may take 1-2 minutes for the full file)...')

# Read only the columns we need — the full file is ~5GB so we skip unused columns
projects_cols = [
    'ID', 'Platform', 'Name', 'Homepage URL', 'Repository URL',
    'Stars', 'Forks', 'Dependent Repositories Count', 'Dependent Projects Count',
    'Latest Release Published At', 'Status'
]

projects_raw = pd.read_csv(
    PROJECTS_CSV,
    usecols=projects_cols,
    low_memory=False
)

print(f'Total rows loaded: {len(projects_raw):,}')
print(f'Platforms present: {projects_raw["Platform"].value_counts().head(10).to_dict()}')

In [ ]:
# Filter to NPM only and clean up column names
npm_packages = (
    projects_raw[projects_raw['Platform'] == 'NPM']
    .copy()
    .rename(columns={
        'ID': 'package_id',
        'Name': 'name',
        'Homepage URL': 'homepage',
        'Repository URL': 'repo_url',
        'Stars': 'stars',
        'Forks': 'forks',
        'Dependent Repositories Count': 'dependent_repos',
        'Dependent Projects Count': 'dependent_packages',
        'Latest Release Published At': 'last_publish',
        'Status': 'status'
    })
    .drop(columns=['Platform'])
    .reset_index(drop=True)
)

# Parse the publish date
npm_packages['last_publish'] = pd.to_datetime(npm_packages['last_publish'], errors='coerce')

# Extract GitHub owner/repo from the repository URL (used later to join with GitHub Archive)
npm_packages['github_slug'] = (
    npm_packages['repo_url']
    .str.extract(r'github\.com/([^/]+/[^/]+?)(?:\.git)?$', expand=False)
    .str.lower()
)

print(f'NPM packages: {len(npm_packages):,}')
print(f'\nWith GitHub repo URL: {npm_packages["github_slug"].notna().sum():,}')
print(f'Missing publish date: {npm_packages["last_publish"].isna().sum():,}')
npm_packages.head(3)

In [ ]:
# Save full NPM packages table
out_path = f'{PROJECT_ROOT}/data/processed/npm_packages.parquet'
npm_packages.to_parquet(out_path, index=False)
print(f'Saved {len(npm_packages):,} rows → {out_path}')

In [ ]:
print('Loading dependencies CSV...')

deps_cols = ['Platform', 'Project Name', 'Dependency Name', 'Dependency Kind', 'Optional Dependency']

deps_raw = pd.read_csv(
    DEPS_CSV,
    usecols=deps_cols,
    low_memory=False
)

# Filter to NPM → NPM edges only
npm_deps = (
    deps_raw[deps_raw['Platform'] == 'NPM']
    .copy()
    .rename(columns={
        'Project Name': 'src_package',       # the package that has a dependency
        'Dependency Name': 'dst_package',    # the package being depended on
        'Dependency Kind': 'dep_kind',       # runtime, dev, peer, etc.
        'Optional Dependency': 'is_optional'
    })
    .drop(columns=['Platform'])
    .reset_index(drop=True)
)

# We only care about runtime and peer dependencies for our graph
# (dev dependencies don't affect production blast radius)
runtime_deps = npm_deps[npm_deps['dep_kind'].isin(['runtime', 'peer', None, float('nan')])]

print(f'Total NPM dependency edges: {len(npm_deps):,}')
print(f'Runtime/peer edges (used for graph): {len(runtime_deps):,}')
print(f'\nDep kind breakdown:\n{npm_deps["dep_kind"].value_counts()}')

In [ ]:
out_path = f'{BASE}/data/processed/npm_dependencies.parquet'
runtime_deps.to_parquet(out_path, index=False)
print(f'Saved {len(runtime_deps):,} dependency edges → {out_path}')

## 3 — GitHub Archive: contributor commit activity

**Why:** The Libraries.io dataset tells us *which* packages exist and how they depend on each other, but it doesn't tell us *who* is actively maintaining them. For the Bus Factor metric, we need to know how many distinct contributors committed to each repo in the last 12 months. The GitHub Archive on BigQuery captures all public GitHub events (pushes, PRs, forks) since 2011.

**Cost awareness:** This query scans ~1-3GB, well within the free 1TB/month quota. The `WHERE` clause limiting to NPM-associated repos and recent years is what keeps it small. Always check the bytes estimate in BigQuery before running a large query — the BigQuery UI shows it before execution, and Colab shows it in the dry-run output below.

**What the SQL does:**
- Joins `github_repos.commits` (commit-level data) filtered to repos we know are NPM packages (via the `github_slug` column we extracted from Libraries.io)
- Counts distinct committers per repo per month
- Filters to 2022-onwards to keep data manageable

In [ ]:
# Build the list of GitHub slugs we care about (repos that correspond to NPM packages)
npm_github_slugs = (
    npm_packages['github_slug']
    .dropna()
    .unique()
    .tolist()
)
print(f'NPM packages with GitHub repos: {len(npm_github_slugs):,}')
print('Sample slugs:', npm_github_slugs[:5])

In [ ]:
# We'll query GitHub Archive via the githubarchive BigQuery public dataset.
# The table is partitioned by date (YYYYMMDD suffix), so we query specific year tables.
#
# Schema we use:
#   githubarchive.year.* — one table per year
#   Each row = one GitHub event
#   type = 'PushEvent' for commits
#   repo.name = 'owner/repo' slug
#   actor.login = the GitHub username who pushed
#   created_at = timestamp

# We pass the slug list to BigQuery as an UNNEST array to avoid a huge IN clause
# BigQuery handles this efficiently as a hash join.

# Format slugs as a BigQuery array literal  e.g. ['facebook/react', 'lodash/lodash', ...]
slug_array = ', '.join(f"'{s}'" for s in npm_github_slugs[:50000])  # cap at 50k

query = f"""
SELECT
  LOWER(repo.name)                          AS github_slug,
  actor.login                               AS author_login,
  DATE(created_at)                          AS commit_date,
  COUNT(*)                                  AS push_count
FROM (
  -- Union the last 3 years of GitHub Archive tables
  SELECT type, repo, actor, created_at FROM `githubarchive.year.2022`
  UNION ALL
  SELECT type, repo, actor, created_at FROM `githubarchive.year.2023`
  UNION ALL
  SELECT type, repo, actor, created_at FROM `githubarchive.year.2024`
)
WHERE
  type = 'PushEvent'
  AND LOWER(repo.name) IN UNNEST([{slug_array}])
GROUP BY
  github_slug, author_login, commit_date
"""

# Dry run first — this estimates bytes scanned WITHOUT charging your quota
job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry_run_job = bq.query(query, job_config=job_config)
bytes_scanned = dry_run_job.total_bytes_processed
print(f'Estimated bytes scanned: {bytes_scanned / 1e9:.2f} GB')
print(f'Estimated cost (on-demand): ${bytes_scanned / 1e12 * 6.25:.4f} (free under 1TB/month)')

In [ ]:
# Run the actual query (will take 1-5 minutes depending on BigQuery slot availability)
print('Running BigQuery job... (this may take a few minutes)')
job_config = bigquery.QueryJobConfig(use_query_cache=True)
query_job = bq.query(query, job_config=job_config)

gh_commits = query_job.to_dataframe()
gh_commits['commit_date'] = pd.to_datetime(gh_commits['commit_date'])

print(f'Rows returned: {len(gh_commits):,}')
print(f'Unique repos: {gh_commits["github_slug"].nunique():,}')
print(f'Unique authors: {gh_commits["author_login"].nunique():,}')
print(f'Date range: {gh_commits["commit_date"].min()} → {gh_commits["commit_date"].max()}')
gh_commits.head(3)

In [ ]:
out_path = f'{BASE}/data/processed/gh_commits.parquet'
gh_commits.to_parquet(out_path, index=False)
print(f'Saved {len(gh_commits):,} rows → {out_path}')

## 4 — OSV: NPM vulnerability feed

**Why:** OSV (Open Source Vulnerabilities) is a distributed vulnerability database maintained by Google. It aggregates CVEs and GitHub Security Advisories into a unified JSON schema. The NPM-specific feed (`osv-vulnerabilities/npm/all.zip`) contains every known CVE affecting an NPM package, with the affected version ranges specified precisely.

We download the full zip and parse each JSON file into a flat DataFrame. Later, in notebook 04, we'll join this against our package graph.

**OSV JSON schema (what we extract):**
```json
{
  "id": "GHSA-xxxx-xxxx-xxxx",
  "affected": [{
    "package": { "name": "lodash", "ecosystem": "npm" },
    "ranges": [{ "type": "SEMVER", "events": [{"introduced": "0"}, {"fixed": "4.17.21"}] }]
  }]
}
```

In [ ]:
import urllib.request
import zipfile
import json
from pathlib import Path

OSV_URL = 'https://osv-vulnerabilities.storage.googleapis.com/npm/all.zip'
OSV_ZIP = f'{BASE}/data/raw/osv_npm.zip'
OSV_DIR = f'{BASE}/data/raw/osv_npm'

print('Downloading OSV npm feed...')
urllib.request.urlretrieve(OSV_URL, OSV_ZIP)

print('Extracting...')
with zipfile.ZipFile(OSV_ZIP, 'r') as z:
    z.extractall(OSV_DIR)

osv_files = list(Path(OSV_DIR).glob('*.json'))
print(f'Extracted {len(osv_files):,} vulnerability JSON files')

In [ ]:
def parse_osv_file(path):
    """Extract the fields we need from one OSV JSON file."""
    with open(path) as f:
        data = json.load(f)

    vuln_id = data.get('id', '')
    summary = data.get('summary', '')
    severity = data.get('database_specific', {}).get('severity', 'UNKNOWN')
    published = data.get('published', '')

    rows = []
    for affected in data.get('affected', []):
        pkg = affected.get('package', {})
        pkg_name = pkg.get('name', '').lower()
        ecosystem = pkg.get('ecosystem', '')

        # Collect affected version ranges as a JSON string (we'll parse them in notebook 04)
        ranges = json.dumps(affected.get('ranges', []))

        rows.append({
            'vuln_id': vuln_id,
            'package_name': pkg_name,
            'ecosystem': ecosystem,
            'severity': severity,
            'summary': summary,
            'published': published,
            'affected_ranges': ranges
        })
    return rows

print('Parsing OSV files...')
all_rows = []
for path in osv_files:
    try:
        all_rows.extend(parse_osv_file(path))
    except Exception as e:
        print(f'  Warning: could not parse {path.name}: {e}')

osv_df = pd.DataFrame(all_rows)
osv_df['published'] = pd.to_datetime(osv_df['published'], errors='coerce')

print(f'Total OSV records: {len(osv_df):,}')
print(f'Unique packages with vulnerabilities: {osv_df["package_name"].nunique():,}')
print(f'Severity breakdown:\n{osv_df["severity"].value_counts()}')
osv_df.head(3)

In [ ]:
out_path = f'{BASE}/data/processed/osv_npm.parquet'
osv_df.to_parquet(out_path, index=False)
print(f'Saved {len(osv_df):,} OSV records → {out_path}')

## 5 — npm registry: maintainer data (Orphan Risk)

**Why:** Libraries.io doesn't track the npm registry's `maintainers` field directly — that's the list of npm accounts who have publish rights to a package. A package can have a GitHub repo with many contributors but only 1 npm maintainer who can actually push a new version. That single account is the real chokepoint for supply chain attacks (as seen in the `event-stream` hijacking).

We fetch this from the npm registry API (`registry.npmjs.org/<pkg>`) for the top 5k packages. To avoid rate-limiting, we use `asyncio` + `aiohttp` with concurrency capped at 20 parallel requests and a short delay.

**What we capture:**
- `maintainers` — list of npm usernames with publish access
- `time.modified` — when the package was last published
- `dist-tags.latest` — the current latest version

In [ ]:
import asyncio
import aiohttp
from tqdm.auto import tqdm

# Select the top 5k NPM packages by dependent package count
# These are the ones that matter most for blast radius
top_packages = (
    npm_packages
    .dropna(subset=['name'])
    .nlargest(5000, 'dependent_packages')
    ['name']
    .tolist()
)
print(f'Fetching npm registry data for {len(top_packages):,} packages')
print('Sample:', top_packages[:5])

In [ ]:
NPM_REGISTRY = 'https://registry.npmjs.org'
CONCURRENCY  = 20    # max parallel requests
DELAY_MS     = 50    # milliseconds between batches

async def fetch_package(session, name, semaphore):
    """Fetch metadata for one npm package. Returns a dict or None on error."""
    # The abbreviated endpoint (?write=true is faster and returns the key fields)
    url = f'{NPM_REGISTRY}/{name}'
    headers = {'Accept': 'application/vnd.npm.install-v1+json'}  # abbreviated manifest
    async with semaphore:
        try:
            async with session.get(url, headers=headers, timeout=aiohttp.ClientTimeout(total=10)) as resp:
                if resp.status != 200:
                    return None
                data = await resp.json(content_type=None)
                maintainers = [m.get('name', '') for m in data.get('maintainers', [])]
                last_modified = data.get('time', {}).get('modified', '')
                latest_version = data.get('dist-tags', {}).get('latest', '')
                return {
                    'name': name,
                    'npm_maintainers': maintainers,
                    'maintainer_count': len(maintainers),
                    'npm_last_modified': last_modified,
                    'latest_version': latest_version
                }
        except Exception:
            return None

async def fetch_all(package_names):
    semaphore = asyncio.Semaphore(CONCURRENCY)
    results = []
    connector = aiohttp.TCPConnector(limit=CONCURRENCY)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [fetch_package(session, name, semaphore) for name in package_names]
        for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc='npm registry'):
            result = await coro
            if result:
                results.append(result)
            await asyncio.sleep(DELAY_MS / 1000)
    return results

# Colab runs an event loop already, so we use nest_asyncio to allow await
import nest_asyncio
nest_asyncio.apply()

print(f'Fetching npm registry data (concurrency={CONCURRENCY}, delay={DELAY_MS}ms)...')
maintainer_records = asyncio.run(fetch_all(top_packages))
print(f'Successfully fetched: {len(maintainer_records):,} / {len(top_packages):,}')

In [ ]:
npm_maintainers = pd.DataFrame(maintainer_records)
npm_maintainers['npm_last_modified'] = pd.to_datetime(npm_maintainers['npm_last_modified'], errors='coerce')

print(f'Total packages fetched: {len(npm_maintainers):,}')
print(f'\nMaintainer count distribution:')
print(npm_maintainers['maintainer_count'].describe())
print(f'\nPackages with only 1 npm maintainer: {(npm_maintainers["maintainer_count"] == 1).sum():,}')
print(f'Packages with ≤2 npm maintainers: {(npm_maintainers["maintainer_count"] <= 2).sum():,}')
npm_maintainers.head(3)

In [ ]:
out_path = f'{BASE}/data/processed/npm_maintainers.parquet'
# Convert list column to JSON string for parquet compatibility
npm_maintainers_save = npm_maintainers.copy()
npm_maintainers_save['npm_maintainers'] = npm_maintainers_save['npm_maintainers'].apply(json.dumps)
npm_maintainers_save.to_parquet(out_path, index=False)
print(f'Saved {len(npm_maintainers):,} rows → {out_path}')

## 6 — Build the 10k development sample

**Why:** The full dataset is large. During development (notebooks 02–04), we iterate quickly on a 10k-package sample so that each cell runs in seconds instead of minutes. Once we're happy with the logic, notebook 06 re-runs everything on the full dataset.

We select the top 10k packages by `dependent_packages` (how many other packages depend on them) — this captures the most structurally important nodes and makes the sample representative of the real risk landscape.

In [ ]:
sample_packages = (
    npm_packages
    .dropna(subset=['name'])
    .nlargest(10_000, 'dependent_packages')
    .reset_index(drop=True)
)

sample_names = set(sample_packages['name'].str.lower())

# Filter dependencies to only those where BOTH src and dst are in our sample
sample_deps = runtime_deps[
    runtime_deps['src_package'].str.lower().isin(sample_names) &
    runtime_deps['dst_package'].str.lower().isin(sample_names)
].reset_index(drop=True)

# Filter commits to repos in our sample
sample_slugs = set(sample_packages['github_slug'].dropna().str.lower())
sample_commits = gh_commits[gh_commits['github_slug'].isin(sample_slugs)].reset_index(drop=True)

print(f'Sample packages:     {len(sample_packages):,}')
print(f'Sample dep edges:    {len(sample_deps):,}')
print(f'Sample commit rows:  {len(sample_commits):,}')

In [ ]:
sample_packages.to_parquet(f'{BASE}/data/sample/npm_packages_10k.parquet', index=False)
sample_deps.to_parquet(f'{BASE}/data/sample/npm_dependencies_10k.parquet', index=False)
sample_commits.to_parquet(f'{BASE}/data/sample/gh_commits_10k.parquet', index=False)
print('Sample files saved to data/sample/')

## 7 — Sanity checks

Run these cells to verify ingestion worked correctly before moving to notebook 02.

In [ ]:
# Check 1: Well-known packages should be in our dataset
known_packages = ['react', 'lodash', 'express', 'typescript', 'axios', 'webpack']
found = [p for p in known_packages if p in sample_names]
missing = [p for p in known_packages if p not in sample_names]

print('CHECK 1 — known packages in sample:')
print(f'  Found:   {found}')
if missing:
    print(f'  MISSING: {missing}  ← investigate if this list is non-empty')
else:
    print('  All known packages present ✓')

In [ ]:
# Check 2: lodash should have CVEs in OSV
lodash_cves = osv_df[osv_df['package_name'] == 'lodash']
print(f'CHECK 2 — lodash CVEs in OSV: {len(lodash_cves)} records')
if len(lodash_cves) > 0:
    print(lodash_cves[['vuln_id', 'severity', 'summary']].to_string())
    print('  lodash has CVEs ✓')
else:
    print('  WARNING: no CVEs found for lodash — check OSV parsing')

In [ ]:
# Check 3: Top packages by dependent count (should be recognisable names)
print('CHECK 3 — top 10 packages by dependent_packages:')
print(
    sample_packages
    .nlargest(10, 'dependent_packages')
    [['name', 'dependent_packages', 'stars', 'last_publish']]
    .to_string(index=False)
)

In [ ]:
# Check 4: npm maintainer data spot-check
print('CHECK 4 — npm maintainer data spot-check:')
spot = npm_maintainers[npm_maintainers['name'].isin(['react', 'lodash', 'express'])]
print(spot[['name', 'maintainer_count', 'npm_last_modified', 'latest_version']].to_string(index=False))

print('\nPackages with exactly 1 npm maintainer (top 10 by dependent_packages):')
solo = (
    npm_maintainers[npm_maintainers['maintainer_count'] == 1]
    .merge(npm_packages[['name', 'dependent_packages']], on='name')
    .nlargest(10, 'dependent_packages')
    [['name', 'dependent_packages', 'maintainer_count', 'npm_last_modified']]
)
print(solo.to_string(index=False))

In [ ]:
# Summary
print('=' * 50)
print('INGESTION COMPLETE — files in Google Drive:')
print('=' * 50)
import os
for root, dirs, files in os.walk(f'{BASE}/data'):
    level = root.replace(f'{BASE}/data', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(files):
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'{indent}  {f}  ({size_mb:.1f} MB)')